In [1]:
# Explicitly Point Spark to the Correct Python Executable
# If you already have numpy installed in a specific environment (like conda or a virtualenv), but PySpark is defaulting to /usr/bin/python3 or system Python, 
# explicitly set the Python environment variables before initializing SparkContext:

import os
import sys

# Set these before initializing SparkContext (sc)
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession. \
    builder. \
    enableHiveSupport(). \
    appName('Window-Function-Demo'). \
    master('yarn'). \
    getOrCreate()

In [4]:
# spark = SparkSession. \
#     builder. \
#     enableHiveSupport(). \
#     appName('Date-Time-Demo'). \
#     master('local[*]'). \
#     getOrCreate()

In [5]:
sc = spark.sparkContext

## Deep dive into Apache Spark Window Functions

Window functions operate on groups of data and return values for each record or group.
We’ll do a Deep Dive into Apache Spark Window Functions

let’s look at what window functions are and when we should use them. We use various functions in Apache Spark like `month` (return month from the date), `round` (round off the value), and `floor`(gives floor value for a given input), etc. which will be performed on each record and will return a value for each record. Then we have various aggregated functions that will be performed on a group of data and return a single value for each group like `sum`, `avg`, `min`, `max`, and `count`.  

 But what if we would like to perform the operation on a group of data and would like to have a single value/result for each record? We can use window functions in such cases. They can define the ranking for records, cumulative distribution, moving average, or identify the records prior to or after the current record.
 
Let’s use some Python API examples to learn about the following window functions:
- **Aggregate:** min, max, avg, count, and sum.  
- **Ranking:** rank, dense_rank, percent_rank, row_num, and ntile
- **Analytical:** cume_dist, lag, and lead
- **Custom boundary:** rangeBetween and rowsBetween

### Create Spark DataFrame

Now let’s create a sample Spark DataFrame which we will use throughout this session. First, let’s load the required libraries.

In [6]:
from pyspark.sql import Window
from pyspark.sql.functions import *

Now we will create the DataFrame with some dummy data which we will use to discuss various window functions.

In [7]:
empList = [("sales", 1, 5000), ("personnel", 2, 3900)\
            , ("sales", 3, 4800), ("sales", 4, 4800)\
            , ("personnel", 5, 3500), ("develop", 7, 4200)\
            , ("develop", 8, 6000), ("develop", 9, 4500)\
            , ("develop", 10, 5200), ("develop", 11, 5200)]
empsalary = spark.createDataFrame(empList, ['depName', 'empNo', 'salary'])
empsalary.show()

+---------+-----+------+
|  depName|empNo|salary|
+---------+-----+------+
|    sales|    1|  5000|
|personnel|    2|  3900|
|    sales|    3|  4800|
|    sales|    4|  4800|
|personnel|    5|  3500|
|  develop|    7|  4200|
|  develop|    8|  6000|
|  develop|    9|  4500|
|  develop|   10|  5200|
|  develop|   11|  5200|
+---------+-----+------+



### Window aggregate functions

Let’s look at some aggregated window functions to see how they work.

First, we need to define the specification of the window. Let’s say we would like to get the aggregated data based on the department. So we will define our window based on the department name (column: `depname`) in this example.

#### Create a window specification for aggregate function

In [8]:
byDepName = Window.partitionBy("depName")
print(type(byDepName))

<class 'pyspark.sql.window.WindowSpec'>


#### Apply aggregate function on window

Now within the department (column: `depName`) we can apply various aggregated functions. So let's try to find the `max` and `min` salary in each department. Here we have selected only desired columns (`depName`, `max_salary`, and `min_salary`) and removed the duplicate records.

In [9]:
agg_sal = empsalary\
.withColumn("max_salary", max("salary").over(byDepName))\
.withColumn("min_salary", min("salary").over(byDepName))
        
agg_sal.select("depName", "max_salary", "min_salary").dropDuplicates().show()        

+---------+----------+----------+
|  depName|max_salary|min_salary|
+---------+----------+----------+
|  develop|      6000|      4200|
|    sales|      5000|      4800|
|personnel|      3900|      3500|
+---------+----------+----------+



In [7]:
# agg_sal = empsalary\
# .withColumn("max_salary", max("salary").over(byDepName))\
# .withColumn("min_salary", min("salary").over(byDepName))
        
# agg_sal.select("depName", "max_salary", "min_salary").show()

Now let’s see how it works. We have partitioned the data on department name:

<img src="images/windowfn1.jpg" align = "left">

Now when we perform the aggregate function, it will be applied to each partition and return the aggregated value (`min` and `max` in our case.).

<img src="images/windowfn2.jpg" align = "left">

### Window ranking functions

In this section, we will discuss several types of ranking functions.

#### Create window specification for ranking function

Now let's say we would like to rank the employees based on their salary within the department. The employee with the highest salary will have rank 1, and the one with the least salary will rank last. Here we will partition the data based on department (column: `depName`) and within the department, we will sort the data based on `salary` in descending order.

In [8]:
winSpec = Window.partitionBy("depName").orderBy(col("salary").desc())
print(type(winSpec))

<class 'pyspark.sql.window.WindowSpec'>


For each department, records are sorted based on salary in descending order.

<img src="images/windowfn3.jpg" align = "left">

#### 1. Rank function: `rank`

This function will return the rank of each record within a partition and skip the subsequent rank following any duplicate rank:

In [9]:
rank_df = empsalary.withColumn("rank", rank().over(winSpec))
rank_df.show()

+---------+-----+------+----+
|  depName|empNo|salary|rank|
+---------+-----+------+----+
|  develop|    8|  6000|   1|
|  develop|   10|  5200|   2|
|  develop|   11|  5200|   2|
|  develop|    9|  4500|   4|
|  develop|    7|  4200|   5|
|    sales|    1|  5000|   1|
|    sales|    3|  4800|   2|
|    sales|    4|  4800|   2|
|personnel|    2|  3900|   1|
|personnel|    5|  3500|   2|
+---------+-----+------+----+



Here we can see some of the ranks are duplicated and some ranks are missing. For example, in `develop` department we have 2 employees with `rank = 2` and no employee with `rank = 3` because the rank function will keep the same rank for the same value and skip the next ranks accordingly.

<img src="images/Rank_Function1.jpg" align = "left">
<img src="images/Rank_Function1-1.jpg" align = "left">
<img src="images/Rank_Function2.jpg" align = "left">
<img src="images/Rank_Function3.jpg" align = "left">

#### 2. Dense rank: `dense_rank`

This function will return the rank of each record within a partition but will not skip any rank.

In [10]:
dense_rank_df = empsalary.withColumn("dense_rank", dense_rank().over(winSpec))
dense_rank_df.show()

+---------+-----+------+----------+
|  depName|empNo|salary|dense_rank|
+---------+-----+------+----------+
|  develop|    8|  6000|         1|
|  develop|   10|  5200|         2|
|  develop|   11|  5200|         2|
|  develop|    9|  4500|         3|
|  develop|    7|  4200|         4|
|    sales|    1|  5000|         1|
|    sales|    3|  4800|         2|
|    sales|    4|  4800|         2|
|personnel|    2|  3900|         1|
|personnel|    5|  3500|         2|
+---------+-----+------+----------+



<img src="images/Dense_Rank1.jpg" align = "left">
<img src="images/Dense_Rank1-1.jpg" align = "left">
<img src="images/Dense_Rank2.jpg" align = "left">
<img src="images/Dense_Rank3.jpg" align = "left">

#### 3. Row num function: `row_number`

This function will assign the row number within the window. If 2 rows will have the same value for ordering column, it is non-deterministic which row number will be assigned to each row with same value.

In [11]:
row_num_df = empsalary.withColumn("row_number", row_number().over(winSpec))

row_num_df.show()

+---------+-----+------+----------+
|  depName|empNo|salary|row_number|
+---------+-----+------+----------+
|  develop|    8|  6000|         1|
|  develop|   10|  5200|         2|
|  develop|   11|  5200|         3|
|  develop|    9|  4500|         4|
|  develop|    7|  4200|         5|
|    sales|    1|  5000|         1|
|    sales|    3|  4800|         2|
|    sales|    4|  4800|         3|
|personnel|    2|  3900|         1|
|personnel|    5|  3500|         2|
+---------+-----+------+----------+



<img src="images/row_number1.jpg" align = "left">
<img src="images/row_number1-1.jpg" align = "left">
<img src="images/row_number2.jpg" align = "left">
<img src="images/row_number3.jpg" align = "left">

#### 4. Percent rank function: `percent_rank`

This function will return the relative (percentile) rank within the partition.

PERCENT_RANK is a statistical window function that calculates the relative standing of a value within a dataset, expressed as a number between 0 and 1.

In [12]:
#winSpec = Window.partitionBy("depName").orderBy(col("salary"))
percent_rank_df = empsalary.withColumn("percent_rank", percent_rank().over(winSpec))
percent_rank_df.show()

+---------+-----+------+------------+
|  depName|empNo|salary|percent_rank|
+---------+-----+------+------------+
|  develop|    8|  6000|         0.0|
|  develop|   10|  5200|        0.25|
|  develop|   11|  5200|        0.25|
|  develop|    9|  4500|        0.75|
|  develop|    7|  4200|         1.0|
|    sales|    1|  5000|         0.0|
|    sales|    3|  4800|         0.5|
|    sales|    4|  4800|         0.5|
|personnel|    2|  3900|         0.0|
|personnel|    5|  3500|         1.0|
+---------+-----+------+------------+



<img src="images/percent_rank1.jpg" align = "left">
<img src="images/percent_rank1-1.jpg" align = "left">
<img src="images/percent_rank2.jpg" align = "left">
<img src="images/percent_rank3.jpg" align = "left">

#### 5. N-tile function: `ntile`

This function can further sub-divide the window into n groups based on a window specification or partition. For example, if we need to divide the departments further into say three groups we can specify ntile as 3.

In data analysis and SQL, NTILE is a highly significant window function used to divide an ordered dataset into a specified number of approximately equal groups or "buckets" (e.g., 4 for quartiles, 10 for deciles, or 100 for percentiles)

In [13]:
#winSpec = Window.partitionBy("depName").orderBy(col("salary").desc())
ntile_df = empsalary.withColumn("ntile", ntile(3).over(winSpec))
ntile_df.show()

+---------+-----+------+-----+
|  depName|empNo|salary|ntile|
+---------+-----+------+-----+
|  develop|    8|  6000|    1|
|  develop|   10|  5200|    1|
|  develop|   11|  5200|    2|
|  develop|    9|  4500|    2|
|  develop|    7|  4200|    3|
|    sales|    1|  5000|    1|
|    sales|    3|  4800|    2|
|    sales|    4|  4800|    3|
|personnel|    2|  3900|    1|
|personnel|    5|  3500|    2|
+---------+-----+------+-----+



<img src="images/ntile_function1.jpg" align = "left">
<img src="images/ntile_function1-1.jpg" align = "left">
<img src="images/ntile_function2.jpg" align = "left">
<img src="images/ntile_function3.jpg" align = "left">

### Window analytical functions

Next, we will discuss the analytical functions like cumulative distribution, lag, and lead.

#### 1. Cumulative distribution function: `cume_dist`

This function gives the cumulative distribution of values for the window/partition.

Define the window specification for analytical functions

In [14]:
winSpec = Window.partitionBy("depName").orderBy(col("salary"))

apply cume_dist function to get the cumulative distribution.

In [15]:
cume_dist_df = empsalary.withColumn("cume_dist",cume_dist().over(winSpec))
cume_dist_df.show()

+---------+-----+------+------------------+
|  depName|empNo|salary|         cume_dist|
+---------+-----+------+------------------+
|  develop|    7|  4200|               0.2|
|  develop|    9|  4500|               0.4|
|  develop|   10|  5200|               0.8|
|  develop|   11|  5200|               0.8|
|  develop|    8|  6000|               1.0|
|    sales|    3|  4800|0.6666666666666666|
|    sales|    4|  4800|0.6666666666666666|
|    sales|    1|  5000|               1.0|
|personnel|    5|  3500|               0.5|
|personnel|    2|  3900|               1.0|
+---------+-----+------+------------------+



<img src="images/cume_dist1.jpg" align = "left">
<img src="images/cume_dist1-1.jpg" align = "left">
<img src="images/cume_dist2.jpg" align = "left">
<img src="images/cume_dist3.jpg" align = "left">

#### 2. Lag function: `lag`

This function will return the value prior to offset rows from DataFrame.

The lag function takes 3 arguments `(lag(col, count = 1, default = None))`,  

`col`: defines the columns on which function needs to be applied.  
`count`: for how many rows we need to look back.  
`default`: defines the default value.

The LAG function is an analytic window function used to retrieve the value of a preceding row in a dataset without needing to join a table to itself. It is highly significant in database management, data analytics, and time-series forecasting for tracking trends and measuring changes over time

In [16]:
#winSpec = Window.partitionBy("depName").orderBy(col("salary"))
lag_df = empsalary.withColumn("lag", lag("salary", 2).over(winSpec))
lag_df.show()

+---------+-----+------+----+
|  depName|empNo|salary| lag|
+---------+-----+------+----+
|  develop|    7|  4200|null|
|  develop|    9|  4500|null|
|  develop|   10|  5200|4200|
|  develop|   11|  5200|4500|
|  develop|    8|  6000|5200|
|    sales|    3|  4800|null|
|    sales|    4|  4800|null|
|    sales|    1|  5000|4800|
|personnel|    5|  3500|null|
|personnel|    2|  3900|null|
+---------+-----+------+----+



<img src="images/lag0.jpg" align = "left">

<img src="images/lag1.jpg" align = "left">
<img src="images/lag1-1.jpg" align = "left">
<img src="images/lag2.jpg" align = "left">
<img src="images/lag3.jpg" align = "left">

#### 3. Lead function: `lead`

This function will return the value after the offset rows from DataFrame.

lead function takes 3 arguments `(lead(col, count = 1, default = None))`  

`col`: defines the columns on which the function needs to be applied.  
`count`: for how many rows we need to look forward/after the current row.  
`default`: defines the default value.

In SQL and data analysis, the LEAD() function is a highly valuable window function that allows you to access data from a subsequent (future) row within the same result set, without needing to perform complex, resource-heavy self-joins

In [17]:
#winSpec = Window.partitionBy("depName").orderBy(col("salary"))
lead_df = empsalary.withColumn("lead", lead("salary", 2).over(winSpec))
lead_df.show()

+---------+-----+------+----+
|  depName|empNo|salary|lead|
+---------+-----+------+----+
|  develop|    7|  4200|5200|
|  develop|    9|  4500|5200|
|  develop|   10|  5200|6000|
|  develop|   11|  5200|null|
|  develop|    8|  6000|null|
|    sales|    3|  4800|5000|
|    sales|    4|  4800|null|
|    sales|    1|  5000|null|
|personnel|    5|  3500|null|
|personnel|    2|  3900|null|
+---------+-----+------+----+



<img src="images/lead0.jpg" align = "left">

<img src="images/lead1-1.jpg" align = "left">


### Custom window definition

By default, the boundaries of the window are defined by partition column and we can specify the ordering via window specification. As an example, for develop department, start of the window is min value of salary, and end of the window is max value of salary.

But what if we would like to change the boundaries of the window? The following functions can be used to define the window within each partition.

#### 1. `rangeBetween`

Using the `rangeBetween` function, we can define the boundaries explicitly.
For example, let’s define the start as 100 and end as 300 units from current salary and see what it means. Start as 100 means the window will start from 100 units and end at 300 value from current value (both start and end values are inclusive).

* Define window specification

In [18]:
winSpec = Window.partitionBy("depName").orderBy("salary").rangeBetween(100, 300)

* Apply custom window specification

In [19]:
range_between_df = empsalary.withColumn("max_salary", max("salary").over(winSpec))
range_between_df.show()

+---------+-----+------+----------+
|  depName|empNo|salary|max_salary|
+---------+-----+------+----------+
|  develop|    7|  4200|      4500|
|  develop|    9|  4500|      null|
|  develop|   10|  5200|      null|
|  develop|   11|  5200|      null|
|  develop|    8|  6000|      null|
|    sales|    3|  4800|      5000|
|    sales|    4|  4800|      5000|
|    sales|    1|  5000|      null|
|personnel|    5|  3500|      null|
|personnel|    2|  3900|      null|
+---------+-----+------+----------+



Let’s try to understand the output now.

For `depname=develop`, `salary =4200`, start of the window will be (current value + start) which is 4200 + 100 = 4300. End of the window will be (current value + end) which is 4200 + 300 = 4500.  

Since there is only one salary value in the range 4300 to 4500 inclusive, which is `4500` for `develop` department, we got `4500` as max_salary for `4200` (check output above).

<img src="images/rangeBetween1.jpg" align = "left">

Similarly for `depname=develop`, `salary = 4500`, the window will be (start : 4500 + 100 = 4600, end : 4500 + 300 = 4800). But there are no salary values in the range 4600 to 4800 inclusive for develop department so max value will be null (check output above).

<img src="images/rangeBetween2.jpg" align = "left">

There are some special boundary values which can be used here.

* `Window.currentRow`: to specify a current value in a row.
* `Window.unboundedPreceding`: This can be used to have an unbounded start for the window.
* `Window.unboundedFollowing`: This can be used to have an unbounded end for the window.

For example, we need to find the max salary which is greater than 300 from employee salary. So we’ll define the start value as 300L and define the end value as `Window.unboundedFollowing`:

In [20]:
winSpec = Window.partitionBy("depName").orderBy("salary")\
.rangeBetween(300, Window.unboundedFollowing)


In [21]:
range_unbounded_df = empsalary.withColumn("max_salary", max("salary").over(winSpec))
range_unbounded_df.show()

+---------+-----+------+----------+
|  depName|empNo|salary|max_salary|
+---------+-----+------+----------+
|  develop|    7|  4200|      6000|
|  develop|    9|  4500|      6000|
|  develop|   10|  5200|      6000|
|  develop|   11|  5200|      6000|
|  develop|    8|  6000|      null|
|    sales|    3|  4800|      null|
|    sales|    4|  4800|      null|
|    sales|    1|  5000|      null|
|personnel|    5|  3500|      3900|
|personnel|    2|  3900|      null|
+---------+-----+------+----------+



So, for `depname = personnel`, `salary = 3500`. the window will be (start : 3500 + 300 = 3800, end : unbounded). So the maximum value in this range is 3900 (check output above).

Similarly, for `depname = sales`, `salary = 4800`, the window will be (start : 4800 + 300, 5100, end : unbounded). Since there are no values greater than 5100 for sales department, `null` results.

#### 2. `rowsBetween`

With `rangeBetween`, we defined the start and end of the window using the value of the ordering column. However, we can also define the start and end of the window with the relative row position.

For example, we would like to create a window where start of the window is one row prior to current and end is one row after current row.

* Define custom window specification

In [22]:
winSpec = Window.partitionBy("depName").orderBy("salary").rowsBetween(-1, 1)

* Apply custom window specification

In [23]:
rows_between_df = empsalary.withColumn("max_salary", max("salary").over(winSpec))
rows_between_df.show()

+---------+-----+------+----------+
|  depName|empNo|salary|max_salary|
+---------+-----+------+----------+
|  develop|    7|  4200|      4500|
|  develop|    9|  4500|      5200|
|  develop|   10|  5200|      5200|
|  develop|   11|  5200|      6000|
|  develop|    8|  6000|      6000|
|    sales|    3|  4800|      4800|
|    sales|    4|  4800|      5000|
|    sales|    1|  5000|      5000|
|personnel|    5|  3500|      3900|
|personnel|    2|  3900|      3900|
+---------+-----+------+----------+



Let’s try to understand the output now.

For `depname = develop`, `salary = 4500`, a window will be defined with one row prior and after the current row (highlighted in green). So salaries within the window are `(4200, 4500, 5200)` and max is `5200` (check output above).

<img src="images/rowsBetween1.jpg" align = "left">

Similarly, for `depname = sales`, `salary =5000`, a window will be defined with one prior and after the current row. Since there are no rows after this row, the window will only have 2 rows (highlighted in green) which have salaries as `(4800, 5000)` and max is `5000` (check output above).

<img src="images/rowsBetween2.jpg" align = "left">

We can also use the special boundaries `Window.unboundedPreceding`, `Window.unboundedFollowing`, and `Window.currentRow` as we did previously with `rangeBetween`.  

Note: Ordering is not necessary with `rowsBetween`, but I have used it to keep the results consistent on each run.